# Stroke Model Training & Comparison
This notebook loads the dataset, applies data cleaning and preprocessing, trains multiple models, compares them on all criteria (Accuracy, Precision, Recall, F1-Score, ROC-AUC), and saves the best pipeline.

In [1]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix
)

# Models to compare
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

import warnings
warnings.filterwarnings('ignore')

In [3]:
BASE_DIR = os.path.abspath('..')
DATA_DIR = os.path.join(BASE_DIR, 'Data')
MODEL_DIR = os.path.join(BASE_DIR, 'Model')
os.makedirs(MODEL_DIR, exist_ok=True)

file_path = os.path.join(DATA_DIR, 'stroke.csv')
df = pd.read_csv(file_path)
print(f"Dataset shape: {df.shape}")
print(f"Class distribution:\n{df['stroke'].value_counts()}\n")
df.head()

Dataset shape: (5110, 12)
Class distribution:
stroke
0    4861
1     249
Name: count, dtype: int64



,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [4]:
# Basic cleaning
df.replace(r'^\s*$', np.nan, regex=True, inplace=True)
df = df.drop(columns=['id'], errors='ignore')


In [6]:
X = df.drop(columns=['stroke'])
y = df['stroke'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])


In [7]:
pipelines = {
    'Logistic Regression': Pipeline([('preprocessor', preprocessor), ('model', LogisticRegression(max_iter=1000, random_state=42))]),
    'Gaussian Naive Bayes': Pipeline([('preprocessor', preprocessor), ('model', GaussianNB())]),
    'Random Forest': Pipeline([('preprocessor', preprocessor), ('model', RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_split=5, min_samples_leaf=2, random_state=42))]),
    'Gradient Boosting': Pipeline([('preprocessor', preprocessor), ('model', GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=4, min_samples_split=5, min_samples_leaf=2, random_state=42))]),
    # 'SVM (RBF)': Pipeline([('preprocessor', preprocessor), ('model', SVC(kernel='rbf', probability=True, random_state=42))]),
    # 'KNN': Pipeline([('preprocessor', preprocessor), ('model', KNeighborsClassifier(n_neighbors=7))]),
    'Decision Tree': Pipeline([('preprocessor', preprocessor), ('model', DecisionTreeClassifier(max_depth=5, min_samples_split=5, min_samples_leaf=2, random_state=42))]),
    'AdaBoost': Pipeline([('preprocessor', preprocessor), ('model', AdaBoostClassifier(n_estimators=100, learning_rate=0.1, random_state=42))]),
}


In [9]:
print('=' * 70)
print('  MODEL COMPARISON RESULTS')
print('=' * 70)

results = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, pipeline in pipelines.items():
    try:
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        if hasattr(pipeline['model'], 'predict_proba'):
            y_proba = pipeline.predict_proba(X_test)[:, 1]
        else:
            y_proba = y_pred
        
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        auc = roc_auc_score(y_test, y_proba) if len(np.unique(y_test)) == 2 else 0
        
        cv_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='accuracy')
        cv_mean = cv_scores.mean()
        cv_std = cv_scores.std()
        
        results.append({
            'Model': name,
            'Accuracy': acc,
            'Precision': prec,
            'Recall': rec,
            'F1-Score': f1,
            'ROC-AUC': auc,
            'CV Mean': cv_mean,
            'CV Std': cv_std,
        })
    except Exception as e:
        print(f"Error training {name}: {e}")


  MODEL COMPARISON RESULTS


In [10]:
results_df = pd.DataFrame(results)
results_df['Composite'] = (
    results_df['F1-Score'] * 0.30 +
    results_df['ROC-AUC'] * 0.30 +
    results_df['Accuracy'] * 0.15 +
    results_df['Precision'] * 0.10 +
    results_df['Recall'] * 0.10 +
    results_df['CV Mean'] * 0.05
)

results_df = results_df.sort_values('Composite', ascending=False).reset_index(drop=True)

print('\n\n' + '=' * 70)
print('  FINAL COMPARISON TABLE (sorted by composite score)')
print('=' * 70)
print(results_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

best_model_name = results_df.iloc[0]['Model']
print(f"\n{'★' * 50}")
print(f"  BEST MODEL: {best_model_name}")
print(f"  Composite Score: {results_df.iloc[0]['Composite']:.4f}")
print(f"{'★' * 50}")




  FINAL COMPARISON TABLE (sorted by composite score)
               Model  Accuracy  Precision  Recall  F1-Score  ROC-AUC  CV Mean  CV Std  Composite
 Logistic Regression    0.9521     1.0000  0.0200    0.0392   0.8417   0.9511  0.0000     0.5566
   Gradient Boosting    0.9491     0.3333  0.0400    0.0714   0.8006   0.9435  0.0043     0.4885
       Random Forest    0.9511     0.0000  0.0000    0.0000   0.8293   0.9513  0.0005     0.4390
            AdaBoost    0.9511     0.0000  0.0000    0.0000   0.8271   0.9513  0.0005     0.4384
       Decision Tree    0.9491     0.0000  0.0000    0.0000   0.8202   0.9496  0.0025     0.4359
Gaussian Naive Bayes    0.2984     0.0641  0.9800    0.1202   0.7860   0.2789  0.0564     0.4350

★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
  BEST MODEL: Logistic Regression
  Composite Score: 0.5566
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★


In [11]:
best_pipeline = pipelines[best_model_name]
y_pred_best = best_pipeline.predict(X_test)

print(f"\n{'=' * 70}")
print(f"  DETAILED REPORT: {best_model_name}")
print(f"{'=' * 70}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_best))

model_path = os.path.join(MODEL_DIR, 'stroke_pipeline.pkl')
joblib.dump(best_pipeline, model_path)
print(f"\n✅ Saved best model ({best_model_name}) pipeline to {model_path}")


  DETAILED REPORT: Logistic Regression

Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.98       972
           1       1.00      0.02      0.04        50

    accuracy                           0.95      1022
   macro avg       0.98      0.51      0.51      1022
weighted avg       0.95      0.95      0.93      1022


✅ Saved best model (Logistic Regression) pipeline to c:\Users\Prince\OneDrive\Desktop\Health Risk Prediction\Model\stroke_pipeline.pkl
